# 05 OLS / Ridge / Lasso / Elastic Net 多因子选股

## 论文参考
- **作者**: Jiao Xu
- **年份**: 2023
- **标题**: *Fundamental Quantitative Investment Research Based on Machine Learning*
- **DOI**: 10.54254/2754-1169/46/20230335
- **摘要**: 本文系统比较了OLS、Ridge、Lasso、Elastic Net四种线性回归方法在多因子量化选股中的表现。实证结果表明，OLS策略年化收益率达到35.96%，正则化方法在不同市场环境下展现出差异化的优势:Ridge通过L2正则化在因子共线性较强时保持稳定，Lasso通过L1正则化实现自动因子筛选，Elastic Net兼顾两者优势。

> **⚠️ 教学演示声明**
> 
> 本 Notebook 为量化策略算法教学样例，回测结果包含**前视偏差 (Look-ahead Bias)**：
> - 训练标签使用了未来收益率（`shift(-N)`）
> - StandardScaler 等预处理可能在全量数据上拟合
> - 模型参数选择可能基于完整样本期
> 
> **所有回测收益率仅供学习参考，不代表策略的实际可期望表现，不可直接用于实盘交易。**

## 算法原理

### 1. 普通最小二乘法 (OLS)
$$\hat{\beta}_{OLS} = \arg\min_{\beta} \| y - X\beta \|_2^2$$

### 2. Ridge 回归 (L2正则化)
$$\hat{\beta}_{Ridge} = \arg\min_{\beta} \| y - X\beta \|_2^2 + \lambda \| \beta \|_2^2$$
- 系数向零收缩但不为零，适用于多重共线性

### 3. Lasso 回归 (L1正则化)
$$\hat{\beta}_{Lasso} = \arg\min_{\beta} \| y - X\beta \|_2^2 + \lambda \| \beta \|_1$$
- 产生稀疏解，自动进行因子筛选

### 4. Elastic Net (L1+L2混合)
$$\hat{\beta}_{EN} = \arg\min_{\beta} \| y - X\beta \|_2^2 + \lambda_1 \| \beta \|_1 + \lambda_2 \| \beta \|_2^2$$
$$= \arg\min_{\beta} \| y - X\beta \|_2^2 + \alpha \lambda \| \beta \|_1 + \frac{(1-\alpha)\lambda}{2} \| \beta \|_2^2$$
- $\alpha$ 控制L1/L2比例; 兼顾变量选择和分组效应

### 策略流程
1. 每月末构建截面因子数据 (15个因子)
2. 使用过去6个月数据训练4种模型
3. 预测下月各股票收益率
4. 选择预测收益率最高的Top 10等权持有
5. 月度调仓，对比4种方法的绩效差异

In [ ]:
# === Cell 3: 正则化路径动画 ===
import numpy as np
import sys
sys.path.insert(0, '..')
from shared.animation_helpers import animate_regularization_path

# 生成模拟因子数据
np.random.seed(42)
n_samples, n_factors = 200, 8
X_demo = np.random.randn(n_samples, n_factors)
true_coef = np.array([3.0, -2.0, 0, 1.5, 0, 0, -0.8, 0.5])
y_demo = X_demo @ true_coef + np.random.randn(n_samples) * 0.5

fig = animate_regularization_path(X_demo, y_demo,
    title="正则化路径: Lasso vs Ridge 系数收缩对比")
fig.show()

In [ ]:
# === Cell 4: 导入与配置 ===
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler

from shared.data_fetcher import get_stock_daily, get_index_daily, get_multiple_stocks_daily
from shared.factors import build_factor_panel, winsorize, standardize
from shared.backtest_engine import MultiStockBacktester
from shared.visualization import (plot_equity_curve, plot_drawdown,
                                   plot_metrics_table, plot_cumulative_comparison)
from shared.constants import *

print("All imports successful.")

In [ ]:
# === Cell 5: 数据获取 ===
STOCK_POOL = [
    "600519", "601318", "600036", "000858", "601166",
    "600276", "601398", "600030", "000333", "002415",
    "600900", "601888", "600809", "000568", "002304",
    "601012", "600031", "603259", "600585", "000661",
]

stock_data = get_multiple_stocks_daily(STOCK_POOL, DEFAULT_START, DEFAULT_END)
benchmark = get_index_daily(CSI300_CODE, DEFAULT_START, DEFAULT_END)

print(f"Successfully loaded {len(stock_data)} stocks")
print(f"Benchmark data: {len(benchmark)} trading days")

In [ ]:
# === Cell 6: 因子工程 ===
panel = build_factor_panel(stock_data)
print(f"Factor panel shape: {panel.shape}")
print(f"Columns: {list(panel.columns)}")

# 选取15个核心因子作为特征
FEATURE_COLS = [
    'mom_5', 'mom_10', 'mom_20', 'mom_60',
    'vol_5', 'vol_10', 'vol_20',
    'price_to_ma_5', 'price_to_ma_10', 'price_to_ma_20',
    'rsi', 'macd_hist', 'bb_pctb', 'bb_width', 'vp_corr',
]
FEATURE_COLS = [c for c in FEATURE_COLS if c in panel.columns]
TARGET_COL = 'fwd_return_20d'

# 截面标准化: 每个日期截面独立标准化
for col in FEATURE_COLS:
    panel[col] = panel.groupby('date')[col].transform(
        lambda x: standardize(winsorize(x))
    )

panel.dropna(subset=FEATURE_COLS + [TARGET_COL], inplace=True)
print(f"After cleaning: {panel.shape}")
print(f"Features used ({len(FEATURE_COLS)}): {FEATURE_COLS}")

In [ ]:
# === Cell 7: 滚动训练 - 4种模型对比 ===
panel['date'] = pd.to_datetime(panel['date'])
panel['year_month'] = panel['date'].dt.to_period('M')
months = sorted(panel['year_month'].unique())

TRAIN_WINDOW = 6  # 6个月训练窗口
TOP_N = 10

model_configs = {
    'OLS': LinearRegression(),
    'Ridge': Ridge(alpha=1.0),
    'Lasso': Lasso(alpha=0.01, max_iter=10000),
    'ElasticNet': ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=10000),
}

selections = {name: {} for name in model_configs}
coef_history = {name: [] for name in model_configs}

for i in range(TRAIN_WINDOW, len(months) - 1):
    # Training data: past TRAIN_WINDOW months
    train_months = months[i - TRAIN_WINDOW:i]
    train_mask = panel['year_month'].isin(train_months)
    train_df = panel[train_mask]

    # Prediction target: next month
    pred_month = months[i]
    pred_mask = panel['year_month'] == pred_month
    pred_df = panel[pred_mask]

    if len(train_df) < 50 or len(pred_df) < 5:
        continue

    X_train = train_df[FEATURE_COLS].values
    y_train = train_df[TARGET_COL].values
    X_pred = pred_df[FEATURE_COLS].values

    # Scale features
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_pred_s = scaler.transform(X_pred)

    # Cross-validate and select best alpha for regularized models
    for name, base_model in model_configs.items():
        try:
            model = type(base_model)(**base_model.get_params())

            # Simple alpha search for regularized models
            if name in ['Ridge', 'Lasso', 'ElasticNet']:
                best_alpha, best_score = 1.0, -np.inf
                for alpha in [0.001, 0.01, 0.1, 1.0, 10.0]:
                    params = model.get_params()
                    params['alpha'] = alpha
                    trial = type(base_model)(**params)
                    cv_scores = cross_val_score(trial, X_train_s, y_train,
                                               cv=3, scoring='neg_mean_squared_error')
                    if cv_scores.mean() > best_score:
                        best_score = cv_scores.mean()
                        best_alpha = alpha
                params = model.get_params()
                params['alpha'] = best_alpha
                model = type(base_model)(**params)

            model.fit(X_train_s, y_train)
            y_pred = model.predict(X_pred_s)

            # Record coefficients
            coefs = model.coef_ if hasattr(model, 'coef_') else np.zeros(len(FEATURE_COLS))
            coef_history[name].append(coefs)

            # Select top N stocks
            pred_df_copy = pred_df.copy()
            pred_df_copy['predicted'] = y_pred
            top_stocks = pred_df_copy.nlargest(TOP_N, 'predicted')['symbol'].tolist()

            # Rebalance date = last trading day of prediction month
            rebal_date = pred_df_copy['date'].max()
            selections[name][rebal_date] = top_stocks

        except Exception as e:
            continue

for name in selections:
    print(f"{name}: {len(selections[name])} rebalance periods")

In [ ]:
# === Cell 8: 回测 ===
all_prices = {sym: stock_data[sym]['close'] for sym in stock_data}
bench_close = benchmark['close'] if 'close' in benchmark.columns else None

results = {}
for name in model_configs:
    bt = MultiStockBacktester(
        initial_capital=INITIAL_CAPITAL,
        rebalance_freq='M'
    )
    result = bt.run(all_prices, selections[name], bench_close)
    results[name] = result
    print(f"\n=== {name} ===")
    for k, v in result['metrics'].items():
        print(f"  {k}: {v}")

In [ ]:
# === Cell 9: 可视化 ===

# 1. 四模型累计收益对比
strategy_returns = {name: results[name]['returns'] for name in results}
plot_cumulative_comparison(strategy_returns, title="OLS / Ridge / Lasso / ElasticNet 累计收益对比")

# 2. 各模型收益曲线 vs 基准
for name in results:
    bench_eq = None
    if results[name].get('benchmark_returns') is not None:
        bench_eq = INITIAL_CAPITAL * (1 + results[name]['benchmark_returns']).cumprod()
    plot_equity_curve(results[name]['equity_curve'], bench_eq,
                      title=f"{name} - 累计收益 vs 沪深300")

# 3. 回撤对比 (选择最佳模型)
best_model = max(results, key=lambda k: float(results[k]['metrics']['年化收益率'].strip('%')))
plot_drawdown(results[best_model]['equity_curve'],
              title=f"最优模型 ({best_model}) - 回撤")

# 4. 绩效指标对比表
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(12, 5))
ax.axis('off')
metrics_keys = ['累计收益率', '年化收益率', '夏普比率', '最大回撤', '胜率']
cell_text = []
for key in metrics_keys:
    row = [key] + [results[m]['metrics'].get(key, 'N/A') for m in model_configs]
    cell_text.append(row)
table = ax.table(cellText=cell_text,
                 colLabels=['指标'] + list(model_configs.keys()),
                 loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 1.8)
for (i, j), cell in table.get_celld().items():
    if i == 0:
        cell.set_facecolor('#2196F3')
        cell.set_text_props(color='white', fontweight='bold')
    elif i % 2 == 0:
        cell.set_facecolor('#f0f0f0')
ax.set_title("四种线性模型绩效对比", fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

# 5. 系数热力图 (最后一期)
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for idx, (name, coefs) in enumerate(coef_history.items()):
    ax = axes[idx // 2, idx % 2]
    if len(coefs) > 0:
        coef_df = pd.DataFrame(coefs, columns=FEATURE_COLS)
        mean_coef = coef_df.mean().sort_values()
        colors = ['#F44336' if v < 0 else '#4CAF50' for v in mean_coef.values]
        ax.barh(mean_coef.index, mean_coef.values, color=colors, alpha=0.8)
    ax.set_title(f"{name} - 平均因子系数", fontsize=12, fontweight='bold')
    ax.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
    ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

## 结果讨论

### 核心发现
1. **OLS**: 无正则化约束，在因子共线性较高时容易过拟合，但在市场因子线性关系明确的时期可获得较高收益
2. **Ridge**: L2正则化保留所有因子但缩小系数，在高共线性环境下最稳定，波动率通常最低
3. **Lasso**: L1正则化产生稀疏解，自动筛除无关因子，模型可解释性最强
4. **Elastic Net**: 兼顾L1和L2优势，适合因子间存在分组结构的场景

### 正则化强度 $\lambda$ 的影响
- $\lambda \to 0$: 退化为OLS，高方差低偏差
- $\lambda \to \infty$: 所有系数趋零，高偏差低方差
- 最优$\lambda$通过交叉验证确定，平衡偏差-方差权衡

### 与论文对比
- Xu (2023) 报告OLS策略年化35.96%，正则化模型在风险调整后表现更优
- 本实验结果可能因股票池和时间窗口不同而有差异
- 正则化方法的核心价值在于提升样本外泛化能力